# 🧅 Módulo 08 — Pipeline y Middleware

> **Objetivo:** Interceptar las invocaciones del agente para logging, métricas, validación, etc.

## ¿Qué es middleware?

Una función que se ejecuta **antes y después** de cada llamada al agente. Permite:

- 📊 Logging y telemetría
- ⏱️ Medición de latencia
- 🔐 Validación de seguridad (bloquear peticiones sensibles)
- 🧪 A/B testing
- 💉 Inyección/transformación de mensajes

## Patrón cebolla

Con varios middleware en `[mw1, mw2, mw3]`, el flujo es:

```
mw1 entra → mw2 entra → mw3 entra → AGENTE → mw3 sale → mw2 sale → mw1 sale
```

## API

```python
from agent_framework import AgentContext

async def my_middleware(context: AgentContext, call_next) -> None:
    # antes
    await call_next()
    # después

agent = Agent(client=..., middleware=[my_middleware])
```


## ⚙️ Setup inicial

Esta celda carga la configuración de Azure OpenAI desde `appsettings.Development.json`
(o variables de entorno) y crea el cliente que reutilizaremos durante todo el notebook.

> 💡 Solo necesitas ejecutar esta celda **una vez** al abrir el notebook.


In [ ]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
# Si abriste el notebook desde la carpeta notebooks/, sube un nivel
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados. Endpoint:", create_chat_client().__class__.__name__)


## 1️⃣ Middleware básico — logging antes y después

Patrón clásico: registrar timestamps de entrada y salida.


In [ ]:
from collections.abc import Awaitable, Callable
from datetime import datetime, timezone
from agent_framework import AgentContext

middleware_logs: list[str] = []


async def logging_middleware(
    context: AgentContext,
    call_next: Callable[[], Awaitable[None]],
) -> None:
    middleware_logs.append(
        f"[BEFORE] Request recibido a las {datetime.now(timezone.utc).strftime('%H:%M:%S')}"
    )
    await call_next()  # ejecuta el agente
    middleware_logs.append(
        f"[AFTER]  Respuesta generada a las {datetime.now(timezone.utc).strftime('%H:%M:%S')}"
    )


agent = Agent(
    client=create_chat_client(),
    instructions="Eres un asistente útil. Responde en una oración.",
    middleware=[logging_middleware],
)

response = await agent.run("¿Cuál es el sentido de la vida?")

print("✅ Logs del middleware:")
for log in middleware_logs:
    print(f"   {log}")
print(f"\n   Respuesta del agente: {response.text}")


## 2️⃣ Cadena de varios middleware (patrón cebolla)

Cada middleware envuelve al siguiente. Útil para componer responsabilidades sin acoplarlas.


In [ ]:
from agent_framework import AgentContext
import time

execution_order: list[str] = []


async def logging_mw(ctx: AgentContext, call_next):
    execution_order.append("1-ENTER: Logging middleware")
    await call_next()
    execution_order.append("1-EXIT:  Logging middleware")


async def timing_mw(ctx: AgentContext, call_next):
    execution_order.append("2-ENTER: Timing middleware")
    start = time.perf_counter()
    await call_next()
    elapsed_ms = (time.perf_counter() - start) * 1000
    execution_order.append(f"2-EXIT:  Timing middleware ({elapsed_ms:.0f}ms)")


async def validation_mw(ctx: AgentContext, call_next):
    execution_order.append("3-ENTER: Validation middleware")
    await call_next()
    execution_order.append("3-EXIT:  Validation middleware")


agent = Agent(
    client=create_chat_client(),
    instructions="Eres un asistente útil. Responde brevemente.",
    middleware=[logging_mw, timing_mw, validation_mw],
)

response = await agent.run("¡Hola!")

print("✅ Pipeline ejecutado (patrón cebolla):")
for i, entry in enumerate(execution_order, start=1):
    print(f"   [{i}] {entry}")
print(f"\n   Respuesta: {response.text}")


## 3️⃣ Middleware que captura metadatos

Puedes leer `context.messages` para inspeccionar lo que llega al agente.
Caso de uso: rastrear el último prompt del usuario para análisis o auditoría.


In [ ]:
captured: dict[str, str | bool] = {"executed": False, "question": ""}


async def metadata_mw(context: AgentContext, call_next):
    captured["executed"] = True
    if context.messages:
        last = context.messages[-1]
        captured["question"] = last.text if hasattr(last, "text") else ""

    print(f"🔍 Middleware capturó: '{captured['question']}'")
    await call_next()
    print("🔍 Middleware: Respuesta completada exitosamente")


agent = Agent(
    client=create_chat_client(),
    instructions="Eres un asistente útil. Siempre responde de forma concisa.",
    middleware=[metadata_mw],
)

response = await agent.run("¿Qué es Python?")

print("\n✅ Metadatos capturados:")
print(f"   Ejecutado: {captured['executed']}")
print(f"   Pregunta:  {captured['question']}")
print(f"   Respuesta: {response.text}")


## 🎯 Resumen

| Capacidad | Cómo se hace |
|-----------|--------------|
| Crear middleware | `async def mw(context: AgentContext, call_next): ...` |
| Lógica antes | Código antes de `await call_next()` |
| Lógica después | Código después de `await call_next()` |
| Encadenar varios | `Agent(..., middleware=[mw1, mw2, mw3])` (patrón cebolla) |

> 💡 También existe **function-invocation middleware** que envuelve cada llamada a un tool individualmente — útil para timing/logging por función.

**Siguiente módulo →** [Módulo 09 — Orchestration Sequential](./09_orchestration_sequential.ipynb)
